# SafeRoute — fine-tune YOLOv8n for fire/smoke (D-Fire)

Run this in Colab with a GPU runtime (Runtime -> Change runtime type -> GPU).
Fills in `runs/fire_smoke/weights/best.pt`, which `scripts/run_camera_demo.py` looks for.


In [ ]:
!pip -q install ultralytics

## 1. Get the D-Fire dataset

Easiest path: export a YOLOv8-format copy from Roboflow Universe and paste the download
snippet it gives you here (it looks like the cell below, with your own API key and dataset).
If you already have `datasets/dfire/` structured as in `training/dfire.yaml`, skip this cell.


In [ ]:
# Example (replace with your own Roboflow export snippet):
# !pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_KEY")
# project = rf.workspace("YOUR_WORKSPACE").project("d-fire")
# dataset = project.version(1).download("yolov8", location="datasets/dfire")


In [ ]:
from pathlib import Path
assert Path("datasets/dfire/images/train").exists(), "point this at your dataset (see cell above)"
assert Path("datasets/dfire/images/val").exists(), "need a val split too"
print("dataset OK")


## 2. Write the data config (mirrors training/dfire.yaml)

In [ ]:
%%writefile dfire.yaml
path: datasets/dfire
train: images/train
val: images/val
names:
  0: fire
  1: smoke


## 3. Train

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
model.train(data="dfire.yaml", epochs=60, imgsz=640, batch=16, project="runs", name="fire_smoke", patience=15, plots=True)


## 4. Check validation metrics (precision/recall per class -> feeds scripts/calibrate_perception.py)

In [ ]:
metrics = model.val()
print(metrics.box.maps)       # mAP50-95 per class
print(metrics.box.p, metrics.box.r)   # precision, recall per class at the default threshold


## 5. Download the weights

Download `runs/fire_smoke/weights/best.pt` and place it in your local repo at the same path
(or point `data/camera_config.json`'s `fire_smoke_weights` at wherever you put it).


In [ ]:
from google.colab import files
files.download("runs/fire_smoke/weights/best.pt")
